In [ ]:
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import urlparse
import io
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

### a) Ler o dataset fakeTelegram.BR_2022.csv, o qual está disponível no link a seguir: https://drive.google.com/file/d/1c_hLzk85pYw-huHSnFYZM_gndUsYRDma/view?usp=drive_link

In [ ]:
fakeTelegram = pd.read_csv("fakeTelegram.BR_2022.csv")

fakeTelegram.head()

### b) Remova os trava-zaps. 

In [ ]:
df = fakeTelegram[fakeTelegram['trava_zap'] == False]

df.head()

### c) Remover as linhas repetidas (duplicadas). 

In [ ]:
df = df.drop_duplicates()
df.head()

### d) Remover textos com menos de 5 palavras

In [ ]:
#Contando as palavras
word_counts = df['text_content_anonymous'].str.split().str.len()

# Criar filtro para textos com mais de 5 palavras
mask_long_texts = word_counts >= 5

# Aplicando o filtro
df = df[mask_long_texts].copy()
df.head()


### e) Para cada atributo numérico apresente: 

#### 1. Medidas de Tendência Central 


In [ ]:
import numpy as np
colunas_numericas_todas = df.select_dtypes(include=np.number).columns

colunas_id = ['id_message', 'dataset_info_id']
colunas_numericas_analise = [col for col in colunas_numericas_todas if col not in colunas_id]
for coluna in colunas_numericas_analise:
    print(f"\n--- Atributo: {coluna} ---")
    
    # Verificar se a coluna existe no DataFrame (segurança extra)
    if coluna not in df.columns:
        print(f"  Coluna '{coluna}' não encontrada no DataFrame.")
        continue
    
    # Contar valores não-nulos
    valores_nao_nulos = df[coluna].notna().sum()
    if valores_nao_nulos == 0:
        print(f"  Todos os valores são NaN. Não é possível calcular medidas.")
        continue
    
    print(f"  Número de valores não-nulos: {valores_nao_nulos} de {len(df)}")
    
    # 1. Média
    media = df[coluna].mean()
    print(f"  Média: {media:.4f}" if pd.notna(media) else "  Média: NaN")
    
    # 2. Mediana
    mediana = df[coluna].median()
    print(f"  Mediana: {mediana:.4f}" if pd.notna(mediana) else "  Mediana: NaN")
    
    # 3. Moda
    moda_series = df[coluna].mode()
    if not moda_series.empty:
        # Formatar floats, manter ints/outros como string
        if df[coluna].dtype == 'float64':
            modas_formatadas = [f"{m:.4f}" for m in moda_series]
            print(f"  Moda(s): {', '.join(modas_formatadas)}")
        else:
            print(f"  Moda(s): {', '.join(moda_series.astype(str))}")
    
        # Frequência da(s) moda(s)
        frequencia_moda = (df[coluna] == moda_series.iloc[0]).sum() # A frequência é a mesma para todas as modas
        print(f"    Frequência da(s) moda(s): {frequencia_moda}")
    else:
        print("  Moda: Nenhuma moda encontrada ou todos os valores são NaN.")
print("="*70)


#### 2. Medidas de Variabilidade 

In [ ]:
for coluna in colunas_numericas_analise:
    print(f"\n--- Atributo: {coluna} ---")

    if coluna not in df.columns:
        print(f"  Coluna '{coluna}' não encontrada no DataFrame.")
        continue

    # Usar apenas valores não-nulos para os cálculos
    dados_coluna = df[coluna].dropna()

    if dados_coluna.empty:
        print(f"  Todos os valores são NaN ou a coluna está vazia. Não é possível calcular medidas de variabilidade.")
        continue
    
    if len(dados_coluna) < 2:
        print(f"  A coluna tem menos de 2 valores não-nulos. Variabilidade não pode ser calculada de forma significativa.")
        min_val = dados_coluna.min()
        max_val = dados_coluna.max()
        if pd.notna(min_val) and pd.notna(max_val):
            amplitude = max_val - min_val
            print(f"  Mínimo: {min_val:.4f}")
            print(f"  Máximo: {max_val:.4f}")
            print(f"  Amplitude (Range): {amplitude:.4f}")
        else:
            print("  Mínimo/Máximo: NaN")
            print("  Amplitude (Range): NaN")
        continue


    print(f"  Número de valores não-nulos considerados: {len(dados_coluna)} de {len(df[coluna])}")

    # 1. Amplitude (Range)
    min_val = dados_coluna.min()
    max_val = dados_coluna.max()
    amplitude = max_val - min_val
    print(f"  Mínimo: {min_val:.4f}")
    print(f"  Máximo: {max_val:.4f}")
    print(f"  Amplitude (Range): {amplitude:.4f}")

    # 2. Variância
    variancia = dados_coluna.var() # Por padrão, ddof=1 (amostral)
    print(f"  Variância (amostral): {variancia:.4f}" if pd.notna(variancia) else "  Variância (amostral): NaN")

    # 3. Desvio Padrão
    desvio_padrao = dados_coluna.std() # Por padrão, ddof=1 (amostral)
    print(f"  Desvio Padrão (amostral): {desvio_padrao:.4f}" if pd.notna(desvio_padrao) else "  Desvio Padrão (amostral): NaN")

    # 4. Intervalo Interquartil (IQR)
    q1 = dados_coluna.quantile(0.25)
    q3 = dados_coluna.quantile(0.75)
    iqr = q3 - q1
    print(f"  Primeiro Quartil (Q1): {q1:.4f}")
    print(f"  Terceiro Quartil (Q3): {q3:.4f}")
    print(f"  Intervalo Interquartil (IQR): {iqr:.4f}" if pd.notna(iqr) else "  Intervalo Interquartil (IQR): NaN")

    # 5. Coeficiente de Variação (CV)
    # CV só faz sentido se a média for diferente de zero e desvio padrão existir.
    media = dados_coluna.mean()
    if pd.notna(desvio_padrao) and pd.notna(media) and media != 0:
        coef_variacao = (desvio_padrao / media) * 100
        print(f"  Coeficiente de Variação (CV): {coef_variacao:.2f}%")
    elif media == 0 and pd.notna(desvio_padrao) and desvio_padrao != 0:
        print("  Coeficiente de Variação (CV): Indefinido (média é zero, desvio padrão não é zero)")
    elif media == 0 and pd.notna(desvio_padrao) and desvio_padrao == 0:
         print("  Coeficiente de Variação (CV): 0.00% (média e desvio padrão são zero)")
    else:
        print("  Coeficiente de Variação (CV): Não pôde ser calculado (média ou desvio padrão é NaN, ou média é zero)")

print("="*70)


#### 4. Boxplot 

In [ ]:
for coluna in colunas_numericas_analise:
    print(f"\n\n--- Atributo: {coluna} ---")

    if coluna not in df.columns:
        print(f"  Coluna '{coluna}' não encontrada no DataFrame.")
        continue

    # Usar apenas valores não-nulos para os cálculos
    dados_coluna = df[coluna].dropna()

    if dados_coluna.empty:
        print(f"  Todos os valores são NaN. Não é possível gerar o boxplot.")
        continue
    
    # Boxplot não faz muito sentido para colunas com um único valor único
    if dados_coluna.nunique() < 2:
        print(f"  A coluna '{coluna}' tem menos de 2 valores únicos. O boxplot não será informativo.")
        continue

    print(f"  Gerando boxplot para {len(dados_coluna)} valores não-nulos.")

    plt.figure(figsize=(8, 6)) # Ajuste o tamanho conforme necessário
    sns.boxplot(y=dados_coluna) # Passa a série diretamente para o eixo y

    plt.title(f'Boxplot para {coluna}')
    plt.ylabel(coluna) # O rótulo do eixo y será o nome da coluna
    plt.grid(axis='y', linestyle='--', alpha=0.7) # Adiciona grade horizontal para melhor leitura
    plt.tight_layout()
    plt.show()

print("="*70)


#### 5. QQ-Plot

In [ ]:
import scipy.stats as stats 

for coluna in colunas_numericas_analise:
    print(f"\n\n--- Atributo: {coluna} ---")

    if coluna not in df.columns:
        print(f"  Coluna '{coluna}' não encontrada no DataFrame.")
        continue

    # Usar apenas valores não-nulos
    dados_coluna = df[coluna].dropna()

    if dados_coluna.empty:
        print(f"  Todos os valores são NaN. Não é possível gerar o QQ-Plot.")
        continue
    
    if dados_coluna.nunique() < 2:
        print(f"  A coluna '{coluna}' tem menos de 2 valores únicos ({dados_coluna.nunique()}). O QQ-Plot não é informativo.")
      
        if dados_coluna.nunique() == 1:
             print(f"  Todos os {len(dados_coluna)} valores não-nulos são idênticos: {dados_coluna.iloc[0]:.4f}")
        continue
  
    if len(dados_coluna) < 3: 
        print(f"  A coluna '{coluna}' tem apenas {len(dados_coluna)} valor(es) não-nulo(s). O QQ-Plot pode não ser muito significativo.")
   

    print(f"  Gerando QQ-Plot para {len(dados_coluna)} valores não-nulos.")

    plt.figure(figsize=(8, 6))
  
    try:
        (_, _, r) = stats.probplot(dados_coluna, dist="norm", plot=plt)
        
        plt.title(f'QQ-Plot de {coluna} vs. Normal (R²={r**2:.4f})') # Exibindo R^2
        plt.xlabel('Quantis Teóricos (Normal)')
        plt.ylabel('Quantis da Amostra Ordenados')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"  Não foi possível gerar o QQ-Plot para {coluna}: {e}")
        # Isso pode acontecer em casos extremos, como variância zero se não for pego antes.


print("="*80)


#### 6. Teste de Normalidade

In [ ]:
# Nível de significância
alfa = 0.05

for coluna in colunas_numericas_analise:
    print(f"\n\n--- Atributo: {coluna} ---")

    if coluna not in df.columns:
        print(f"  Coluna '{coluna}' não encontrada no DataFrame.")
        continue

    # Usar apenas valores não-nulos
    dados_coluna = df[coluna].dropna()

    if dados_coluna.empty:
        print(f"  Todos os valores são NaN. Não é possível realizar testes de normalidade.")
        continue
    
    # Verificar se há variância nos dados (necessário para os testes)
    if dados_coluna.nunique() < 2:
        print(f"  A coluna '{coluna}' não possui variância (todos os valores são idênticos ou há apenas um valor). Testes de normalidade não aplicáveis.")
        continue

    print(f"  Analisando {len(dados_coluna)} valores não-nulos.")

    # 1. Teste de Shapiro-Wilk
    # Requer pelo menos 3 amostras.
    if len(dados_coluna) >= 3:
        try:
            stat_shapiro, p_shapiro = stats.shapiro(dados_coluna)
            print(f"\n  Teste de Shapiro-Wilk:")
            print(f"    Estatística: {stat_shapiro:.4f}")
            print(f"    P-valor: {p_shapiro:.4f}")
            if p_shapiro <= alfa:
                print(f"    Conclusão (alfa={alfa}): Rejeita H0. Os dados provavelmente NÃO seguem uma distribuição normal.")
            else:
                print(f"    Conclusão (alfa={alfa}): Não rejeita H0. Não há evidência suficiente para dizer que os dados NÃO seguem uma distribuição normal.")
        except Exception as e:
            print(f"  Não foi possível realizar o teste de Shapiro-Wilk para {coluna}: {e}")
    else:
        print("\n  Teste de Shapiro-Wilk: Não realizado (requer pelo menos 3 amostras).")


    if len(dados_coluna) >= 8: # Mínimo recomendado pela documentação para evitar NaNs
        try:
            stat_dagostino, p_dagostino = stats.normaltest(dados_coluna)
            print(f"\n  Teste de D'Agostino's K-squared:")
            print(f"    Estatística: {stat_dagostino:.4f}")
            print(f"    P-valor: {p_dagostino:.4f}")
            if p_dagostino <= alfa:
                print(f"    Conclusão (alfa={alfa}): Rejeita H0. Os dados provavelmente NÃO seguem uma distribuição normal.")
            else:
                print(f"    Conclusão (alfa={alfa}): Não rejeita H0. Não há evidência suficiente para dizer que os dados NÃO seguem uma distribuição normal.")
        except Exception as e:
            print(f"  Não foi possível realizar o teste de D'Agostino's K-squared para {coluna}: {e}")

    elif len(dados_coluna) > 0: # Se tem dados, mas menos que 8
        print(f"\n  Teste de D'Agostino's K-squared: Não realizado (requer pelo menos 8 amostras, idealmente >20; esta coluna tem {len(dados_coluna)}).")
    else: 
        print("\n  Teste de D'Agostino's K-squared: Não realizado (sem dados suficientes).")


print("="*80)


### f) Para cada par de atributos numéricos apresente:

#### 1. O Coeficiente de Correlação apropriado 

In [ ]:
from itertools import combinations 

pares_de_colunas = list(combinations(colunas_numericas_analise, 2))

print(f"Atributos numéricos para análise de pares: {colunas_numericas_analise}")
print(f"Número de pares a serem analisados: {len(pares_de_colunas)}\n")

for col1, col2 in pares_de_colunas:
    print(f"--- Par: ({col1}) vs ({col2}) ---")

   
    dados_par = df[[col1, col2]].dropna()

    if len(dados_par) < 3:
        print(f"  Não há dados suficientes (mínimo 3 observações com ambos não-nulos) para o par ({col1}, {col2}).")
        print(f"  Observações válidas para o par: {len(dados_par)}")
        print("") # Linha em branco para separar
        continue
    
    # 1. Coeficiente de Correlação de Pearson (r)
    try:
        if dados_par[col1].var() < 1e-9 or dados_par[col2].var() < 1e-9: # Variância muito pequena ou zero
            print("  Correlação de Pearson (r): Não calculada (variância zero ou próxima de zero em um ou ambos os atributos).")
        else:
            pearson_corr, pearson_p_value = stats.pearsonr(dados_par[col1], dados_par[col2])
            print(f"  Correlação de Pearson (r): {pearson_corr:+.4f} (p-valor: {pearson_p_value:.4f})")
            if pearson_p_value <= 0.05:
                print("    Significância (Pearson): A correlação linear é estatisticamente significativa.")
            else:
                print("    Significância (Pearson): A correlação linear NÃO é estatisticamente significativa.")
    except Exception as e:
        print(f"  Erro ao calcular Pearson para ({col1}, {col2}): {e}")

    # 2. Coeficiente de Correlação de Spearman (rho)
    try:
         if dados_par[col1].nunique() < 2 or dados_par[col2].nunique() < 2: # Se não há variabilidade nos ranks
            print("  Correlação de Spearman (rho): Não calculada (um ou ambos os atributos têm valores constantes após remover NaNs).")
         else:
            spearman_corr, spearman_p_value = stats.spearmanr(dados_par[col1], dados_par[col2])
            print(f"  Correlação de Spearman (rho): {spearman_corr:+.4f} (p-valor: {spearman_p_value:.4f})")
            if spearman_p_value <= 0.05:
                print("    Significância (Spearman): A correlação monotônica é estatisticamente significativa.")
            else:
                print("    Significância (Spearman): A correlação monotônica NÃO é estatisticamente significativa.")
    except Exception as e:
        print(f"  Erro ao calcular Spearman para ({col1}, {col2}): {e}")
    print("")

print("\n--- Considerações sobre qual coeficiente é 'mais apropriado' ---")
print("1. Relação Linear vs. Monotônica:")
print("   - Pearson (r): Melhor para relações lineares.")
print("   - Spearman (rho): Melhor para relações monotônicas (que podem ser lineares ou não).")
print("2. Sensibilidade a Outliers e Distribuição dos Dados:")
print("   - Pearson: Sensível a outliers e assume (idealmente) dados aproximadamente normais.")
print("   - Spearman: Robusto a outliers e não assume normalidade (baseado em ranks).")
print("3. Interpretação Prática:")
print("   - Se os dados são bem comportados e a relação parece linear (verifique o scatter plot), Pearson é uma boa escolha.")
print("   - Se houver suspeita de outliers, não normalidade, ou uma relação não linear mas monotônica, Spearman é geralmente mais seguro e apropriado.")
print("   - Se os valores de Pearson e Spearman forem muito diferentes, isso sugere a presença de outliers ou uma relação não linear monotônica, favorecendo a interpretação de Spearman.")
print("   - O p-valor indica se a correlação observada é estatisticamente significativa (ou seja, improvável de ter ocorrido apenas por acaso, dado um nível de significância como 0.05).")

print("="*90)


#### 2. Um Gráfico de Dispersão 

In [ ]:
pares_de_colunas = list(combinations(colunas_numericas_analise, 2))

print(f"Atributos numéricos para análise de pares: {colunas_numericas_analise}")
print(f"Número de pares a serem analisados: {len(pares_de_colunas)}\n")


limite_scatter_individual = 15
pares_plotados_count = 0

for col1, col2 in pares_de_colunas:

    print(f"--- Gráfico de Dispersão: ({col1}) vs ({col2}) ---")


    dados_par = df[[col1, col2]].dropna()

    if len(dados_par) < 1: 
        print(f"  Não há dados (ambos não-nulos) para o par ({col1}, {col2}). Gráfico não gerado.")
        print("")
        continue
    elif len(dados_par) < 10: 
        print(f"  Aviso: Apenas {len(dados_par)} pontos de dados para o par ({col1}, {col2}).")


    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=dados_par[col1], y=dados_par[col2], alpha=0.7, edgecolor='k', s=50)
    
    plt.title(f'Gráfico de Dispersão: {col1} vs {col2}')
    plt.xlabel(col1)
    plt.ylabel(col2)
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.show()
    print("") # Linha em branco para separar os gráficos

    pares_plotados_count += 1

if pares_plotados_count == 0 and len(pares_de_colunas) > 0:
     print("Nenhum gráfico de dispersão foi gerado (verifique se há dados válidos nos pares).")

print("="*90)

### g) Para cada par de atributos categóricos apresente:

#### 1. O resultado do método V de Cramer 

### h) Crie uma visualização (gráfico) para apresentar:

#### 1. As quantidades de grupos, usuários e mensagens; 

In [ ]:
num_grupos = 0
if 'id_group_anonymous' in df.columns:
    num_grupos = df['id_group_anonymous'].nunique()

num_usuarios = 0
if 'id_member_anonymous' in df.columns:
    num_usuarios = df['id_member_anonymous'].nunique()
    
num_mensagens = len(df)

# Preparar dados para o gráfico
labels = ['Grupos Únicos', 'Usuários Únicos', 'Total de Mensagens']
quantidades = [num_grupos, num_usuarios, num_mensagens]

# Criar o gráfico de barras
plt.figure(figsize=(8, 6))


bar_plot = plt.bar(labels, quantidades, color=['#1f77b4', '#ff7f0e', '#2ca02c']) # Cores padrão do Matplotlib

plt.title('Quantidades de Grupos, Usuários e Mensagens', fontsize=15, pad=20)
plt.ylabel('Quantidade', fontsize=12)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

# Adicionar os valores no topo de cada barra
for i, v in enumerate(quantidades):
    offset = max(quantidades) * 0.01 # Pequeno offset relativo
    plt.text(i, v + offset, str(v), color='black', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Ajustar o limite do eixo y para dar espaço ao texto
plt.ylim(0, max(quantidades) * 1.15) 

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout() 
plt.show()

print("="*70)


#### 2. A quantidade de mensagens que possuem apenas texto X mídia;

In [ ]:
media_counts = df['has_media'].value_counts()

count_sem_media = media_counts.get(False, 0) # has_media == False (Apenas Texto)
count_com_media = media_counts.get(True, 0)  
labels = ['Apenas Texto\n(has_media=False)', 'Com Mídia\n(has_media=True)']
quantidades = [count_sem_media, count_com_media]

# Criar o gráfico de barras
plt.figure(figsize=(7, 6)) # Ajustei o tamanho para melhor visualização com rótulos maiores
bar_plot = plt.bar(labels, quantidades, color=['skyblue', 'lightcoral'])
plt.show()

#### 3. Quantidade de mensagens por tipo de mídia (jpg, mp4 etc);

In [ ]:
# Filtrar mensagens que possuem mídia
df_com_media = df[df['has_media'] == True]

tipo_media_counts = df_com_media['message_type'].value_counts()

labels = tipo_media_counts.index.tolist()
quantidades = tipo_media_counts.values.tolist()

# Criar o gráfico de barras
# Ajustar o tamanho da figura com base no número de tipos de mídia
num_tipos = len(labels)
fig_width = max(7, num_tipos * 1.2) # Largura mínima de 7, aumenta com mais tipos
plt.figure(figsize=(fig_width, 6))

colors = sns.color_palette("viridis", n_colors=num_tipos) if num_tipos > 0 else ['skyblue']

bar_plot = plt.bar(labels, quantidades, color=colors)
        
# Adicionar títulos e rótulos essenciais
plt.title('Quantidade de Mensagens por Tipo de Mídia', fontsize=14)
plt.ylabel('Quantidade de Mensagens', fontsize=12)
plt.xlabel('Tipo de Mídia', fontsize=12)
plt.xticks(rotation=45, ha="right") # Rotacionar rótulos do eixo x se forem muitos
plt.tight_layout() # Ajustar layout para evitar sobreposições
plt.show()


#### 4. A relação entre a quantidade de mensagens e a quantidade de palavras presente nas mensagens; 

In [ ]:
word_counts_series = df['text_content_anonymous'].str.split().str.len()
    

word_counts_series = word_counts_series.dropna()


plt.figure(figsize=(10, 6))


sns.histplot(word_counts_series, kde=False, bins='auto', color='steelblue') 


plt.title('Distribuição da Contagem de Palavras por Mensagem', fontsize=14)
plt.xlabel('Número de Palavras na Mensagem', fontsize=12)
plt.ylabel('Quantidade de Mensagens (Frequência)', fontsize=12)



plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import duckdb

nome_arquivo_duckdb = 'df.duckdb'
nome_tabela_duckdb = 'mensagens'

# Conecta ao DuckDB (cria o arquivo se não existir)
conn = duckdb.connect(database=nome_arquivo_duckdb, read_only=False)

# Insere o DataFrame no DuckDB
conn.execute(f"CREATE OR REPLACE TABLE {nome_tabela_duckdb} AS SELECT * FROM df")

#conn.close()

print(f"Os dados foram exportados com sucesso para a tabela '{nome_tabela_duckdb}' no DuckDB: {nome_arquivo_duckdb}")

#### 10. As 30 URLs que mais se repetem (mais compartilhadas); 

In [ ]:
import matplotlib.pyplot as plt

top_30_urls_raw = conn.execute("""
    SELECT media_url, COUNT(*) AS quantidade
    FROM mensagens
    WHERE media_url IS NOT NULL AND media_url != ''
    GROUP BY media_url
    ORDER BY quantidade DESC
    LIMIT 30;
""").fetchall()


df_top_urls = pd.DataFrame(top_30_urls_raw, columns=['URL', 'Quantidade'])


df_top_urls_sorted = df_top_urls.sort_values(by='Quantidade', ascending=True)

labels = df_top_urls_sorted['URL'].tolist()
quantidades = df_top_urls_sorted['Quantidade'].tolist()

num_urls_plot = len(labels)
fig_height = max(6, num_urls_plot * 0.4) 

plt.figure(figsize=(10, fig_height))


colors = sns.color_palette("viridis", n_colors=num_urls_plot)

plt.barh(labels, quantidades, color=colors)

# Rótulos e Título
plt.title(f'Top {num_urls_plot} URLs Mais Compartilhadas', fontsize=14)
plt.xlabel('Número de Compartilhamentos', fontsize=12)
plt.ylabel('URL', fontsize=12)

# Melhorar a legibilidade dos rótulos de URL
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)

# Adicionar os valores ao lado de cada barra para clareza
for index, value in enumerate(quantidades):
    # Calcula um pequeno offset para posicionar o texto ao lado da barra
    offset = (max(quantidades) * 0.02) if quantidades else 0.05 # Evita divisão por zero se quantidades for vazio
    plt.text(value + offset, index, str(value), va='center', fontsize=9)

# Ajustar o limite do eixo X para dar espaço aos rótulos de valor
if quantidades:
    plt.xlim(0, max(quantidades) * 1.20) # Aumenta um pouco o limite para caber o texto
else:
    plt.xlim(0, 1) # Limite padrão se não houver dados

plt.tight_layout(pad=1.0) # Ajusta o layout para evitar sobreposição
plt.show()

#### 11. Os 30 domínios que mais se repetem (mais compartilhados);

In [ ]:
def extract_domain(url):
    try:
        parsed_url = urlparse(url)
        return parsed_url.netloc if parsed_url.netloc else None
    except:
        return None

df['domain'] = df['media_url'].apply(extract_domain)

# Conecta ao DuckDB e cria a tabela temporária (se ainda não estiver conectado)
conn = duckdb.connect(database=':memory:', read_only=False)
conn.execute("CREATE OR REPLACE TEMP TABLE mensagens AS SELECT * FROM df")


top_30_domains_raw = conn.execute("""
    SELECT domain, COUNT(*) AS quantidade
    FROM mensagens
    WHERE domain IS NOT NULL AND domain != ''
    GROUP BY domain
    ORDER BY quantidade DESC
    LIMIT 30;
""").fetchall()


df_top_domains = pd.DataFrame(top_30_domains_raw, columns=['Domínio', 'Quantidade'])


df_top_domains_sorted = df_top_domains.sort_values(by='Quantidade', ascending=True)

labels = df_top_domains_sorted['Domínio'].tolist()
quantidades = df_top_domains_sorted['Quantidade'].tolist()

num_domains_plot = len(labels)
fig_height = max(6, num_domains_plot * 0.4)

plt.figure(figsize=(10, fig_height)) 


colors = sns.color_palette("viridis_r", n_colors=num_domains_plot) # 'viridis_r' inverte a paleta 'viridis'

plt.barh(labels, quantidades, color=colors)

# Rótulos e Título do gráfico
plt.title(f'Top {num_domains_plot} Domínios Mais Compartilhados', fontsize=14)
plt.xlabel('Número de Compartilhamentos', fontsize=12)
plt.ylabel('Domínio', fontsize=12)

# Melhorar a legibilidade dos rótulos
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)


for index, value in enumerate(quantidades):
    offset = (max(quantidades) * 0.02) if quantidades else 0.05 
    plt.text(value + offset, index, str(value), va='center', fontsize=9)


if quantidades:
    plt.xlim(0, max(quantidades) * 1.20) 
else:
    plt.xlim(0, 1)

plt.tight_layout(pad=1.0)
plt.show()



#### 12. Os 30 usuários mais ativos; 

In [ ]:
# --- 2. Obter os 30 usuários mais ativos do DuckDB ---
top_30_usuarios_raw = conn.execute("""
    SELECT id_member_anonymous, COUNT(*) AS quantidade
    FROM mensagens
    WHERE id_member_anonymous IS NOT NULL AND id_member_anonymous != ''
    GROUP BY id_member_anonymous
    ORDER BY quantidade DESC
    LIMIT 30;
""").fetchall()

# 3. Converte a lista de tuplas em um DataFrame pandas
df_top_usuarios = pd.DataFrame(top_30_usuarios_raw, columns=['Usuário', 'Quantidade de Mensagens'])

df_top_usuarios_sorted = df_top_usuarios.sort_values(by='Quantidade de Mensagens', ascending=True)

labels = df_top_usuarios_sorted['Usuário'].tolist()
quantidades = df_top_usuarios_sorted['Quantidade de Mensagens'].tolist()


num_usuarios_plot = len(labels)

fig_height = max(6, num_usuarios_plot * 0.4)

plt.figure(figsize=(10, fig_height)) 


colors = sns.color_palette("rocket", n_colors=num_usuarios_plot) # 'rocket' é uma boa paleta para rankings

plt.barh(labels, quantidades, color=colors)

# Rótulos e Título do gráfico
plt.title(f'Top {num_usuarios_plot} Usuários Mais Ativos', fontsize=14)
plt.xlabel('Número de Mensagens Enviadas', fontsize=12)
plt.ylabel('Usuário', fontsize=12)

# Melhorar a legibilidade dos rótulos
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)

# Adicionar os valores exatos ao lado de cada barra
for index, value in enumerate(quantidades):
    offset = (max(quantidades) * 0.02) if quantidades else 0.05
    plt.text(value + offset, index, str(value), va='center', fontsize=9)

# Ajustar o limite do eixo X para dar espaço aos rótulos de valor
if quantidades:
    plt.xlim(0, max(quantidades) * 1.20)
else:
    plt.xlim(0, 1)

plt.tight_layout(pad=1.0) # Ajusta o layout para evitar sobreposição
plt.show()

#### 13. Relação entre quantidade de mensagens contendo somente texto e mensagens com tendo mídia dos usuários mais ativos: 

In [ ]:

top_30_usuarios_ids = conn.execute("""
    SELECT id_member_anonymous
    FROM mensagens
    WHERE id_member_anonymous IS NOT NULL AND id_member_anonymous != ''
    GROUP BY id_member_anonymous
    ORDER BY COUNT(*) DESC
    LIMIT 30;
""").fetchall()

top_usuarios_list = [user[0] for user in top_30_usuarios_ids]

if not top_usuarios_list:
    print("Não há usuários ativos para analisar a relação entre texto e mídia.")
else:
    # --- 3. Contar mensagens de texto e mídia para esses usuários ---
    df_top_users = df[df['id_member_anonymous'].isin(top_usuarios_list)].copy()

    # Contar mensagens de texto
    text_counts = df_top_users[df_top_users['message_type'] == 'Texto'] \
                  .groupby('id_member_anonymous').size().rename('texto').reset_index()

    # Contar mensagens com mídia
    media_counts = df_top_users[(df_top_users['has_media'] == True) | (df_top_users['has_media_url'] == True)] \
                   .groupby('id_member_anonymous').size().rename('midia').reset_index()

    user_message_types = pd.merge(text_counts, media_counts, on='id_member_anonymous', how='outer')
    user_message_types = user_message_types.fillna(0) # Preenche NaN com 0 para usuários sem um tipo de mensagem
    user_message_types[['texto', 'midia']] = user_message_types[['texto', 'midia']].astype(int)

    df_plot = user_message_types.set_index('id_member_anonymous').loc[top_usuarios_list].reset_index()

    # --- 4. Plotar o gráfico de barras agrupadas ---
    df_plot_melted = df_plot.melt(id_vars='id_member_anonymous', var_name='Tipo de Mensagem', value_name='Quantidade')

    # Ajusta a altura da figura dinamicamente
    num_usuarios_plot = len(df_plot_melted['id_member_anonymous'].unique())
    fig_height = max(8, num_usuarios_plot * 0.5)

    plt.figure(figsize=(12, fig_height)) # Tamanho da figura

    sns.barplot(
        x='Quantidade',
        y='id_member_anonymous',
        hue='Tipo de Mensagem', # Cria barras separadas para cada tipo de mensagem
        data=df_plot_melted,
        palette={'texto': 'skyblue', 'midia': 'lightcoral'} # Cores personalizadas
    )

    plt.title('Mensagens de Texto vs. Mídia pelos Usuários Mais Ativos', fontsize=16)
    plt.xlabel('Número de Mensagens', fontsize=12)
    plt.ylabel('Usuário', fontsize=12)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.legend(title='Tipo de Mensagem', bbox_to_anchor=(1.05, 1), loc='upper left') # Legenda fora do gráfico
    plt.tight_layout()
    plt.show()

#### 14. Os 30 usuários que mais compartilharam texto;

In [ ]:
top_30_users_text_raw = conn.execute("""
    SELECT id_member_anonymous, COUNT(*) AS quantidade
    FROM mensagens
    WHERE id_member_anonymous IS NOT NULL
      AND id_member_anonymous != ''
      AND message_type = 'Texto' -- <--- This is the key filter
    GROUP BY id_member_anonymous
    ORDER BY quantidade DESC
    LIMIT 30;
""").fetchall()

df_top_users_text = pd.DataFrame(top_30_users_text_raw, columns=['Usuário', 'Quantidade de Mensagens de Texto'])

# 4. Preparando dados para plotar no grafico
df_top_users_text_sorted = df_top_users_text.sort_values(by='Quantidade de Mensagens de Texto', ascending=True)

labels = df_top_users_text_sorted['Usuário'].tolist()
quantities = df_top_users_text_sorted['Quantidade de Mensagens de Texto'].tolist()

# --- 5. Cria um grafico de barras horizontais ---
num_users_plot = len(labels)

fig_height = max(6, num_users_plot * 0.4)

plt.figure(figsize=(10, fig_height)) # Set figure size

colors = sns.color_palette("viridis", n_colors=num_users_plot) # 'viridis' is a good sequential palette

plt.barh(labels, quantities, color=colors)


plt.title(f'Top {num_users_plot} Usuários que Mais Compartilharam Texto', fontsize=14)
plt.xlabel('Número de Mensagens de Texto Enviadas', fontsize=12)
plt.ylabel('Usuário', fontsize=12)

# Improve label readability
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)


for index, value in enumerate(quantities):
    offset = (max(quantities) * 0.02) if quantities else 0.05
    plt.text(value + offset, index, str(value), va='center', fontsize=9)


if quantities:
    plt.xlim(0, max(quantities) * 1.20)
else:
    plt.xlim(0, 1)

plt.tight_layout(pad=1.0) # Adjust layout to prevent overlap
plt.show()

#### 15. Os 30 usuários que mais compartilharam mídias;

In [ ]:
top_30_users_media_raw = conn.execute("""
    SELECT id_member_anonymous, COUNT(*) AS quantidade
    FROM mensagens
    WHERE id_member_anonymous IS NOT NULL
      AND id_member_anonymous != ''
      AND has_media = TRUE -- <--- This is the key filter for media
    GROUP BY id_member_anonymous
    ORDER BY quantidade DESC
    LIMIT 30;
""").fetchall()

# Converter a lista de tuplas em um DataFrame pandas
df_top_users_media = pd.DataFrame(top_30_users_media_raw, columns=['Usuário', 'Quantidade de Mídias Compartilhadas'])


df_top_users_media_sorted = df_top_users_media.sort_values(by='Quantidade de Mídias Compartilhadas', ascending=True)

labels = df_top_users_media_sorted['Usuário'].tolist()
quantities = df_top_users_media_sorted['Quantidade de Mídias Compartilhadas'].tolist()

# Criar o gráfico de barras horizontais
num_users_plot = len(labels)
# Ajusta a altura da figura dinamicamente para acomodar todas as barras
fig_height = max(6, num_users_plot * 0.4)

plt.figure(figsize=(10, fig_height)) # Define o tamanho da figura


colors = sns.color_palette("magma", n_colors=num_users_plot)

plt.barh(labels, quantities, color=colors)

# Rótulos e Título do gráfico
plt.title(f'Top {num_users_plot} Usuários que Mais Compartilharam Mídias', fontsize=14)
plt.xlabel('Número de Mídias Compartilhadas', fontsize=12)
plt.ylabel('Usuário', fontsize=12)

# Melhorar a legibilidade dos rótulos
plt.yticks(fontsize=10)
plt.xticks(fontsize=10)

# Adicionar os valores exatos ao lado de cada barra
for index, value in enumerate(quantities):
    offset = (max(quantities) * 0.02) if quantities else 0.05
    plt.text(value + offset, index, str(value), va='center', fontsize=9)

# Ajustar o limite do eixo X para dar espaço aos rótulos de valor
if quantities:
    plt.xlim(0, max(quantities) * 1.20)
else:
    plt.xlim(0, 1)

plt.tight_layout(pad=1.0) # Ajusta o layout para evitar sobreposição
plt.show()

#### 16. As 30 mensagens mais compartilhadas;

In [ ]:
top_30_mensagens_raw = conn.execute("""
    SELECT text_content_anonymous, COUNT(*) AS quantidade
    FROM mensagens
    WHERE text_content_anonymous IS NOT NULL
      AND TRIM(text_content_anonymous) != ''
      AND message_type = 'Texto'
    GROUP BY text_content_anonymous
    ORDER BY quantidade DESC
    LIMIT 30;
""").fetchall()

# 3. Converter a lista de tuplas em um DataFrame pandas
df_top_mensagens = pd.DataFrame(top_30_mensagens_raw, columns=['Mensagem', 'Quantidade de Compartilhamentos'])

# --- IMPORTANTE: Convertendo as mensagens para string explicitamente e TRUNCANDO ---
# Define um limite de caracteres para os rótulos do eixo Y
MAX_LABEL_LENGTH = 70 # Ajuste este valor se precisar de textos mais curtos/longos

df_top_mensagens['Mensagem'] = df_top_mensagens['Mensagem'].astype(str).apply(
    lambda x: x[:MAX_LABEL_LENGTH] + '...' if len(x) > MAX_LABEL_LENGTH else x
)


# 4. Preparar os dados para o gráfico
df_top_mensagens_sorted = df_top_mensagens.sort_values(by='Quantidade de Compartilhamentos', ascending=True)

labels = df_top_mensagens_sorted['Mensagem'].tolist()
quantities = df_top_mensagens_sorted['Quantidade de Compartilhamentos'].tolist()

# --- 5. Criar o gráfico de barras horizontais ---
num_messages_plot = len(labels)
# Ajusta a altura da figura dinamicamente, dando MUITO mais espaço
# Aumentei o fator de multiplicação para dar espaço para textos maiores e mais rótulos
fig_height = max(15, num_messages_plot * 1.2) # Aumentado de 0.9 para 1.2 (pode ajustar)

plt.figure(figsize=(16, fig_height)) # Aumentei a largura da figura para 16

# Usar uma paleta de cores do Seaborn
colors = sns.color_palette("cividis", n_colors=num_messages_plot)

plt.barh(labels, quantities, color=colors)

# Rótulos e Título do gráfico
plt.title(f'Top {num_messages_plot} Mensagens de Texto Mais Compartilhadas', fontsize=18, pad=20) # Fonte do título maior, e pad para afastamento
plt.xlabel('Número de Compartilhamentos', fontsize=16) # Fonte do rótulo do eixo X maior
plt.ylabel('Mensagem (Truncada)', fontsize=16) # Fonte do rótulo do eixo Y maior

# --- Melhorar a legibilidade dos rótulos do eixo Y (mensagens) ---
plt.yticks(fontsize=12) # Mantive 12, pois o truncamento ajuda mais
plt.xticks(fontsize=12) # Aumentei a fonte dos ticks do eixo X também

# Adicionar os valores exatos ao lado de cada barra
for index, value in enumerate(quantities):
    # Ajustei o offset e o fontsize
    offset = (max(quantities) * 0.02) if quantities else 0.05
    plt.text(value + offset, index, str(value), va='center', ha='left', fontsize=11) # Fonte um pouco maior

# Ajustar o limite do eixo X para dar espaço aos rótulos de valor
if quantities:
    plt.xlim(0, max(quantities) * 1.40) # Aumentei ainda mais o limite
else:
    plt.xlim(0, 1)

# Ajustar o layout, talvez com um pouco mais de espaço
plt.tight_layout(pad=2.0) # Aumentei o padding do tight_layout
plt.show()

#### 17. As 30 mensagens mais compartilhadas em grupos diferentes; 

In [ ]:

top_30_mensagens_grupos_raw = conn.execute("""
    SELECT
        text_content_anonymous,
        COUNT(DISTINCT id_group_anonymous) AS num_grupos_distintos
    FROM mensagens
    WHERE text_content_anonymous IS NOT NULL
      AND TRIM(text_content_anonymous) != ''
      AND message_type = 'Texto' -- Focando em mensagens de texto
      AND id_group_anonymous IS NOT NULL -- Garantir que o grupo não é nulo
      AND id_group_anonymous != '' -- E que o grupo não é vazio
    GROUP BY text_content_anonymous
    ORDER BY num_grupos_distintos DESC
    LIMIT 30;
""").fetchall()


df_top_mensagens_grupos = pd.DataFrame(
    top_30_mensagens_grupos_raw,
    columns=['Mensagem', 'Número de Grupos Distintos']
)


MAX_LABEL_LENGTH = 70 # Limite de caracteres para os rótulos do eixo Y

df_top_mensagens_grupos['Mensagem'] = df_top_mensagens_grupos['Mensagem'].astype(str).apply(
    lambda x: x[:MAX_LABEL_LENGTH] + '...' if len(x) > MAX_LABEL_LENGTH else x
)

# 4. Preparar os dados para o gráfico (ordenar para que a mais frequente fique no topo)
df_top_mensagens_grupos_sorted = df_top_mensagens_grupos.sort_values(
    by='Número de Grupos Distintos', ascending=True
)

labels = df_top_mensagens_grupos_sorted['Mensagem'].tolist()
quantities = df_top_mensagens_grupos_sorted['Número de Grupos Distintos'].tolist()


num_messages_plot = len(labels)

fig_height = max(15, num_messages_plot * 1.2)

plt.figure(figsize=(16, fig_height))


colors = sns.color_palette("viridis", n_colors=num_messages_plot) # Mudei para viridis para variar

plt.barh(labels, quantities, color=colors)

# Rótulos e Título do gráfico
plt.title(f'Top {num_messages_plot} Mensagens Compartilhadas em Diferentes Grupos', fontsize=18, pad=20)
plt.xlabel('Número de Grupos Distintos', fontsize=16)
plt.ylabel('Mensagem (Truncada)', fontsize=16)


plt.yticks(fontsize=12)
plt.xticks(fontsize=12)


for index, value in enumerate(quantities):
    offset = (max(quantities) * 0.03) if quantities else 0.05
    plt.text(value + offset, index, str(value), va='center', ha='left', fontsize=11)


if quantities:
    plt.xlim(0, max(quantities) * 1.40)
else:
    plt.xlim(0, 1)

plt.tight_layout(pad=2.0)
plt.show()

#### 18. Mensagens idênticas compartilhadas pelo mesmo usuário (e suas quantidades);

In [ ]:
repeated_messages_by_user_raw = conn.execute("""
    SELECT
        id_member_anonymous,
        text_content_anonymous,
        COUNT(*) AS quantidade_repeticoes
    FROM mensagens
    WHERE id_member_anonymous IS NOT NULL
      AND id_member_anonymous != ''
      AND text_content_anonymous IS NOT NULL
      AND TRIM(text_content_anonymous) != ''
      AND message_type = 'Texto' -- Focando em mensagens de texto
    GROUP BY id_member_anonymous, text_content_anonymous
    HAVING COUNT(*) > 1 -- Apenas as mensagens que foram repetidas (mais de 1 vez)
    ORDER BY quantidade_repeticoes DESC
    LIMIT 30; -- Mostraremos os 30 casos de maior repetição
""").fetchall()

# 3. Converter a lista de tuplas em um DataFrame pandas
df_repeated_messages_by_user = pd.DataFrame(
    repeated_messages_by_user_raw,
    columns=['Usuário', 'Mensagem Repetida', 'Quantidade de Repetições']
)

# --- Formatando a mensagem para melhor visualização (truncando se for muito longa) ---
MAX_DISPLAY_LENGTH = 100 # Limite de caracteres para exibição
df_repeated_messages_by_user['Mensagem Repetida'] = df_repeated_messages_by_user['Mensagem Repetida'].astype(str).apply(
    lambda x: x[:MAX_DISPLAY_LENGTH] + '...' if len(x) > MAX_DISPLAY_LENGTH else x
)

print("--- Top 30 Mensagens Idênticas Repetidas Pelo Mesmo Usuário ---")
if not df_repeated_messages_by_user.empty:
    # Definindo a largura máxima da coluna para melhor exibição
    pd.set_option('display.max_colwidth', None)
    print(df_repeated_messages_by_user.to_string(index=False)) # Usar to_string para garantir que todas as linhas sejam exibidas
else:
    print("Nenhuma mensagem idêntica foi encontrada compartilhada pelo mesmo usuário (ou não há dados suficientes na simulação).")


#### 19. Mensagens idênticas compartilhadas pelo mesmo usuário em grupos distintos (e suas quantidades); 


In [ ]:

repeated_messages_distinct_groups_raw = conn.execute("""
    WITH user_message_groups AS (
        SELECT
            id_member_anonymous,
            text_content_anonymous,
            id_group_anonymous
        FROM mensagens
        WHERE id_member_anonymous IS NOT NULL
          AND id_member_anonymous != ''
          AND text_content_anonymous IS NOT NULL
          AND TRIM(text_content_anonymous) != ''
          AND message_type = 'Texto'
          AND id_group_anonymous IS NOT NULL
          AND id_group_anonymous != ''
        GROUP BY id_member_anonymous, text_content_anonymous, id_group_anonymous
    )
    SELECT
        id_member_anonymous,
        text_content_anonymous,
        COUNT(DISTINCT id_group_anonymous) AS numero_grupos_distintos,
        COUNT(*) AS total_repeticoes_por_usuario_e_mensagem
    FROM user_message_groups
    GROUP BY id_member_anonymous, text_content_anonymous
    HAVING COUNT(DISTINCT id_group_anonymous) > 1
    ORDER BY numero_grupos_distintos DESC, total_repeticoes_por_usuario_e_mensagem DESC
    LIMIT 30;
""").fetchall()

# 3. Converter a lista de tuplas em um DataFrame pandas
df_repeated_messages_distinct_groups = pd.DataFrame(
    repeated_messages_distinct_groups_raw,
    columns=['Usuário', 'Mensagem Repetida', 'Grupos Distintos', 'Total Repetições']
)


MAX_TABLE_DISPLAY_LENGTH = 150 # Ajuste este valor conforme necessário

df_repeated_messages_distinct_groups['Mensagem Repetida'] = (
    df_repeated_messages_distinct_groups['Mensagem Repetida'].astype(str).apply(
        lambda x: x[:MAX_TABLE_DISPLAY_LENGTH] + '...'
        if len(x) > MAX_TABLE_DISPLAY_LENGTH else x
    )
)

print("--- Top 30 Mensagens Idênticas Compartilhadas Pelo Mesmo Usuário em Grupos Distintos ---")
print("\nEsta tabela mostra as mensagens de texto que foram mais replicadas por um mesmo usuário em diferentes grupos, indicando padrões de disseminação de conteúdo.\n")

if not df_repeated_messages_distinct_groups.empty:
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_rows', None)

    print(df_repeated_messages_distinct_groups.to_string(index=False))
else:
    print("Nenhuma mensagem idêntica foi encontrada compartilhada pelo mesmo usuário em grupos distintos (ou não há dados suficientes na simulação).")


#### 20. Os 30 unigramas, bigramas e trigramas mais compartilhados (após a remoção de stop words);

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

# --- 2. Extrair todas as mensagens de texto válidas do DuckDB ---
text_messages_raw = conn.execute("""
    SELECT text_content_anonymous
    FROM mensagens
    WHERE text_content_anonymous IS NOT NULL
      AND TRIM(text_content_anonymous) != ''
      AND message_type = 'Texto';
""").fetchall()

# Achatar a lista de tuplas para uma lista simples de strings
all_messages = [msg[0] for msg in text_messages_raw]

# --- 3. Preparação para Análise de N-gramas ---
# Carregar stop words em português
stop_words_pt = set(stopwords.words('portuguese'))

def preprocess_text(text):
    """Limpa o texto: minúsculas, remove não-alfanuméricos, tokeniza, remove stop words."""
    text = text.lower()
    text = re.sub(r'[^a-záàâãéêíóôõúüç\s]', '', text) # Remove tudo que não é letra (incluindo acentos) ou espaço
    words = word_tokenize(text)
    words = [word for word in words if word.isalpha() and word not in stop_words_pt]
    return words

def generate_ngrams(words, n):
    """Gera n-gramas a partir de uma lista de palavras."""
    return [' '.join(words[i:i+n]) for i in range(len(words) - n + 1)]

all_processed_words = []
for message in all_messages:
    all_processed_words.extend(preprocess_text(message))

# --- 4. Geração e Contagem de N-gramas ---
unigrams = all_processed_words
bigrams = generate_ngrams(all_processed_words, 2)
trigrams = generate_ngrams(all_processed_words, 3)

# Contar frequências
unigram_counts = Counter(unigrams)
bigram_counts = Counter(bigrams)
trigram_counts = Counter(trigrams)

# Converter para DataFrames para plotagem
df_unigrams = pd.DataFrame(unigram_counts.most_common(30), columns=['Unigrama', 'Frequência'])
df_bigrams = pd.DataFrame(bigram_counts.most_common(30), columns=['Bigrama', 'Frequência'])
df_trigrams = pd.DataFrame(trigram_counts.most_common(30), columns=['Trigrama', 'Frequência'])

# --- 5. Funções de Plotagem ---
def plot_ngrams(df, title, x_label, y_label, color_palette="viridis"):
    """Função genérica para plotar n-gramas."""
    if df.empty:
        print(f"Não há dados suficientes para plotar: {title}")
        return

    # Ordenar para que o mais frequente fique no topo do gráfico horizontal
    df_sorted = df.sort_values(by='Frequência', ascending=True)

    labels = df_sorted.iloc[:, 0].tolist() # Pega a primeira coluna (Unigrama/Bigrama/Trigrama)
    quantities = df_sorted['Frequência'].tolist()

    num_entries_plot = len(labels)
    fig_height = max(8, num_entries_plot * 0.6) # Ajuste a altura com base no número de itens

    plt.figure(figsize=(12, fig_height))
    colors = sns.color_palette(color_palette, n_colors=num_entries_plot)
    plt.barh(labels, quantities, color=colors)

    plt.title(title, fontsize=16, pad=15)
    plt.xlabel(x_label, fontsize=12)
    plt.ylabel(y_label, fontsize=12)

    plt.yticks(fontsize=10) # Ajuste o tamanho da fonte dos rótulos do eixo Y
    plt.xticks(fontsize=10)

    # Adicionar os valores exatos ao lado de cada barra
    for index, value in enumerate(quantities):
        offset = (max(quantities) * 0.01) if quantities else 0.05
        plt.text(value + offset, index, str(value), va='center', ha='left', fontsize=9, color='black')

    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.xlim(0, max(quantities) * 1.15) # Ajuste o limite X para dar espaço aos números
    plt.tight_layout(pad=1.5)
    plt.show()

# --- 6. Gerar os Gráficos ---

# Gráfico de Unigramas
plot_ngrams(
    df_unigrams,
    'Top 30 Unigramas Mais Compartilhados (Após Stop Words)',
    'Frequência',
    'Unigrama',
    'plasma' # Outra paleta de cores interessante
)

# Gráfico de Bigramas
plot_ngrams(
    df_bigrams,
    'Top 30 Bigramas Mais Compartilhados (Após Stop Words)',
    'Frequência',
    'Bigrama',
    'magma' # Outra paleta de cores
)

# Gráfico de Trigramas
plot_ngrams(
    df_trigrams,
    'Top 30 Trigramas Mais Compartilhados (Após Stop Words)',
    'Frequência',
    'Trigrama',
    'cividis' # Mais uma opção de paleta de cores
)

In [ ]:
import textwrap # Importa textwrap para quebrar linhas de texto

# --- Configuração de Fonte para Suporte a Emojis/Caracteres Especiais ---
# Tenta definir uma fonte que geralmente suporta uma gama maior de caracteres, incluindo emojis.
# 'Arial' é uma boa tentativa inicial.
# Se estiver no Windows e quiser um suporte robusto a emojis, 'Segoe UI Emoji' pode ser melhor,
# mas nem sempre funciona bem para texto geral junto com emojis.
try:
    plt.rcParams['font.family'] = ['Arial'] # Tenta definir Arial como a fonte principal
    plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans'] # Adiciona 'Arial' à lista de fontes sans-serif
    # Para sistemas Windows, 'Segoe UI Emoji' é uma excelente opção para emojis,
    # mas pode precisar de ajustes adicionais se o texto principal for afetado.
    # Se 'Arial' não funcionar, tente descomentar a linha abaixo e rodar novamente:
    # plt.rcParams['font.family'] = ['Segoe UI Emoji', 'Arial', 'DejaVu Sans']

except Exception as e:
    print(f"Não foi possível configurar a fonte. Erro: {e}")
    print("Verifique se a fonte está instalada no seu sistema.")


# --- 4. Consulta SQL para obter as 30 mensagens distintas mais positivas ---
query = """
SELECT
    DISTINCT text_content_anonymous,
    score_sentiment
FROM
    mensagens
ORDER BY
    score_sentiment DESC
LIMIT 30
"""

top_positive_messages = conn.execute(query).fetchdf()

# --- 6. Gerar o gráfico de barras ---
if not top_positive_messages.empty:
    top_positive_messages = top_positive_messages.sort_values(by='score_sentiment', ascending=True)

    # --- Função para quebrar linhas longas ---
    def wrap_labels(labels, width=70):
        return [textwrap.fill(label, width=width) for label in labels]

    # Aplica a quebra de linha nas mensagens que serão usadas como rótulos do eixo Y.
    wrapped_messages = wrap_labels(top_positive_messages['text_content_anonymous'])

    # --- Ajustes para legibilidade do gráfico e espaçamento entre as barras ---
    line_height_factor = 0.45
    line_height_padding = 0.2

    total_text_height = sum(
        (label.count('\n') + 1) * line_height_factor + line_height_padding
        for label in wrapped_messages
    )

    fig_height = total_text_height + 4

    plt.figure(figsize=(18, fig_height))

    plt.barh(wrapped_messages, top_positive_messages['score_sentiment'], color='skyblue')

    # --- Aumentar o tamanho da fonte dos rótulos ---
    # Define o tamanho da fonte para os rótulos do eixo Y (mensagens com quebra de linha).
    plt.yticks(fontsize=11) # Removido o 'pad' daqui
    # O 'pad' é configurado globalmente para o eixo Y com tick_params
    plt.tick_params(axis='y', pad=10) # Adiciona espaço entre o rótulo do eixo Y e o eixo/barra

    # Define o tamanho da fonte para os rótulos do eixo X (score de sentimento).
    plt.xticks(fontsize=14)
    # Define o tamanho da fonte para o título e os rótulos dos eixos.
    plt.xlabel('Score de Sentimento', fontsize=16)
    plt.ylabel('Mensagem', fontsize=16)
    plt.title('Top 30 Mensagens Mais Positivas (Distintas)\n(Texto quebrado em linhas)', fontsize=18)

    plt.xlim(0, 1.0)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout(pad=1.5)

    plt.show()

    # --- Salvar o Gráfico em Alta Resolução (recomendado para melhor visualização) ---
    plt.savefig('top_mensagens_positivas_legivel.png', dpi=300, bbox_inches='tight')
    print("Gráfico salvo como 'top_mensagens_positivas_legivel.png' em alta resolução.")

else:
    print("Nenhuma mensagem positiva distinta encontrada para gerar o gráfico.")



#### 22. As 30 mensagens mais negativas (distintas);

In [ ]:
# --- 4. Consulta SQL para obter as 30 mensagens distintas mais negativas ---
query = """
SELECT
    DISTINCT text_content_anonymous,
    score_sentiment
FROM
    mensagens
ORDER BY
    score_sentiment ASC
LIMIT 30
"""

top_negative_messages = conn.execute(query).fetchdf()

# --- 6. Gerar o gráfico de barras ---
if not top_negative_messages.empty:
    top_negative_messages = top_negative_messages.sort_values(by='score_sentiment', ascending=True)

    # --- Função para quebrar linhas longas ---
    def wrap_labels(labels, width=50):
        return [textwrap.fill(label, width=width) for label in labels]

    # Aplica a quebra de linha nas mensagens que serão usadas como rótulos do eixo Y.
    wrapped_messages = wrap_labels(top_negative_messages['text_content_anonymous'])

    # --- Ajustes para legibilidade do gráfico e espaçamento entre as barras ---
    line_height_factor = 0.2
    # Diminua este valor para diminuir o espaçamento entre as linhas do texto.
    line_height_padding = 0.1 # Reduzido de 0.2 para 0.1

    total_text_height = sum(
        (label.count('\n') + 1) * line_height_factor + line_height_padding
        for label in wrapped_messages
    )

    fig_height = total_text_height + 4

    plt.figure(figsize=(10, fig_height))

    plt.barh(wrapped_messages, top_negative_messages['score_sentiment'], color='salmon')

    # --- Aumentar o tamanho da fonte dos rótulos ---
    plt.yticks(fontsize=10)
    plt.tick_params(axis='y', pad=2) # Aplica o espaçamento aos rótulos do eixo Y

    plt.xticks(fontsize=10)
    plt.xlim(0, 1.0) # Mantém a escala de 0 a 1

    plt.xlabel('Score de Sentimento', fontsize=16)
    plt.ylabel('Mensagem', fontsize=16)
    plt.title('Top 30 Mensagens Mais Negativas (Distintas)\n(Texto quebrado em linhas)', fontsize=18)

    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout(pad=1.5)

    plt.show()

    # --- Salvar o Gráfico em Alta Resolução ---
    plt.savefig('top_mensagens_negativas_legivel.png', dpi=300, bbox_inches='tight')
    print("Gráfico salvo como 'top_mensagens_negativas_legivel.png' em alta resolução.")

else:
    print("Nenhuma mensagem negativa distinta encontrada para gerar o gráfico.")


#### 23. O usuário mais otimista; 

In [ ]:

query_otimista = """
SELECT
    id_member_anonymous,
    AVG(score_sentiment) AS media_sentimento
FROM
    mensagens
GROUP BY
    id_member_anonymous
ORDER BY
    media_sentimento DESC
LIMIT 1
"""

usuario_mais_otimista = conn.execute(query_otimista).fetchdf()

if not usuario_mais_otimista.empty:
    nome_usuario = usuario_mais_otimista.iloc[0]['id_member_anonymous']
    media_sentimento = usuario_mais_otimista.iloc[0]['media_sentimento']
    print(f"O usuário mais otimista é: **{nome_usuario}** com uma média de sentimento de **{media_sentimento:.2f}**.")
else:
    print("Não foi possível determinar o usuário mais otimista.")

query_todos_usuarios = """
SELECT
    id_member_anonymous,
    AVG(score_sentiment) AS media_sentimento
FROM
    mensagens
GROUP BY
    id_member_anonymous
ORDER BY
    media_sentimento DESC
"""
todos_usuarios_sentimento = conn.execute(query_todos_usuarios).fetchdf()

if not todos_usuarios_sentimento.empty:
    # Ordena para o gráfico (do menor para o maior score para barh)
    todos_usuarios_sentimento = todos_usuarios_sentimento.sort_values(by='media_sentimento', ascending=True)

    # --- Ajustes para tamanho "normal" do gráfico ---
    # Altura base por barra reduzida, pois não há quebra de linha de mensagens longas
    fig_height = len(todos_usuarios_sentimento) * 0.7 + 2

    plt.figure(figsize=(10, fig_height))


    y_pos = range(len(todos_usuarios_sentimento['id_member_anonymous'])) 
    plt.barh(y_pos, todos_usuarios_sentimento['media_sentimento'], color='lightgreen') 

    # Define os rótulos do eixo Y usando as posições numéricas e os nomes dos usuários
    plt.yticks(y_pos, todos_usuarios_sentimento['id_member_anonymous'], fontsize=10)
    plt.tick_params(axis='y', pad=5)

    plt.xticks(fontsize=10)
    plt.xlabel('Média do Score de Sentimento', fontsize=12)
    plt.ylabel('ID do Usuário', fontsize=12)
    plt.title('Média de Sentimento por Usuário (Ordenado por Otimismo)', fontsize=14)

    plt.xlim(0, 1.0)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout(pad=1.0)

    plt.show()
else:
    print("Não há dados de sentimento por usuário para gerar o gráfico.")


#### 24. O usuário mais pessimista; 


In [ ]:
query_pessimista = """
SELECT
    id_member_anonymous,
    AVG(score_sentiment) AS media_sentimento
FROM
    mensagens
GROUP BY
    id_member_anonymous
ORDER BY
    media_sentimento ASC
LIMIT 1
"""

usuario_mais_pessimista = conn.execute(query_pessimista).fetchdf()

# --- 5. Exibir o usuário mais pessimista ---
if not usuario_mais_pessimista.empty:
    nome_usuario = usuario_mais_pessimista.iloc[0]['id_member_anonymous']
    media_sentimento = usuario_mais_pessimista.iloc[0]['media_sentimento']
    print(f"O usuário mais pessimista é: **{nome_usuario}** com uma média de sentimento de **{media_sentimento:.2f}**.")
else:
    print("Não foi possível determinar o usuário mais pessimista.")

query_todos_usuarios_pessimista = """
SELECT
    id_member_anonymous,
    AVG(score_sentiment) AS media_sentimento
FROM
    mensagens
GROUP BY
    id_member_anonymous
ORDER BY
    media_sentimento ASC
"""
todos_usuarios_sentimento_pessimista = conn.execute(query_todos_usuarios_pessimista).fetchdf()

if not todos_usuarios_sentimento_pessimista.empty:
    # Ordena para o gráfico (do menor para o maior score para barh)
    todos_usuarios_sentimento_pessimista = todos_usuarios_sentimento_pessimista.sort_values(by='media_sentimento', ascending=True)

    # --- Ajustes para tamanho "normal" do gráfico ---
    fig_height = len(todos_usuarios_sentimento_pessimista) * 0.7 + 2

    plt.figure(figsize=(10, fig_height))

    y_pos = range(len(todos_usuarios_sentimento_pessimista['id_member_anonymous']))
    plt.barh(y_pos, todos_usuarios_sentimento_pessimista['media_sentimento'], color='lightcoral') # Cor mudada para 'lightcoral' para pessimismo

    plt.yticks(y_pos, todos_usuarios_sentimento_pessimista['id_member_anonymous'], fontsize=10)
    plt.tick_params(axis='y', pad=5)

    plt.xticks(fontsize=10)
    plt.xlabel('Média do Score de Sentimento', fontsize=12)
    plt.ylabel('ID do Usuário', fontsize=12)
    plt.title('Média de Sentimento por Usuário (Ordenado por Pessimismo)', fontsize=14)

    plt.xlim(0, 1.0)
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout(pad=1.0)

    plt.show()
else:
    print("Não há dados de sentimento por usuário para gerar o gráfico.")


#### 25. As 30 maiores mensagens;

In [ ]:
# --- 4. Consulta SQL para obter as 30 mensagens mais longas (distintas) ---
query = """
SELECT
    DISTINCT text_content_anonymous,
    LENGTH(text_content_anonymous) AS message_length
FROM
    mensagens
ORDER BY
    message_length DESC
LIMIT 30
"""

top_longest_messages = conn.execute(query).fetchdf()

# --- 5. Gerar o gráfico de barras ---
if not top_longest_messages.empty:
    # Ordena as mensagens para que a mais longa fique no topo do gráfico de barras horizontais
    top_longest_messages = top_longest_messages.sort_values(by='message_length', ascending=True)

    # --- Função para quebrar linhas longas ---
    def wrap_labels(labels, width=70):
        return [textwrap.fill(label, width=width) for label in labels]

    # Aplica a quebra de linha nas mensagens que serão usadas como rótulos do eixo Y.
    wrapped_messages = wrap_labels(top_longest_messages['text_content_anonymous'])

    # --- Ajustes para legibilidade do gráfico ---
    line_height_factor = 0.45
    line_height_padding = 0.1

    # Calcula a altura necessária para a figura
    total_text_height = sum(
        (label.count('\n') + 1) * line_height_factor + line_height_padding
        for label in wrapped_messages
    )
    fig_height = total_text_height + 4

    # --- Depuração: Verificar a altura calculada da figura ---
    print(f"Altura calculada da figura (fig_height): {fig_height:.2f}")

    # Use plt.subplots() para obter um controle mais explícito sobre a figura e os eixos
    fig, ax = plt.subplots(figsize=(18, fig_height))

    # Cria posições numéricas para as barras
    y_pos = range(len(wrapped_messages))

    # Cria o gráfico de barras horizontais
    # --- NOVO: Aumentado o parâmetro 'height' para diminuir o espaçamento entre as barras ---
    ax.barh(y_pos, top_longest_messages['message_length'], color='darkblue', height=0.9) # Alterado de 0.6 para 0.9

    # Define os rótulos do eixo Y usando as posições numéricas e as mensagens quebradas
    ax.set_yticks(y_pos)
    ax.set_yticklabels(wrapped_messages, fontsize=11)
    ax.tick_params(axis='y', pad=10) # Padding para afastar o texto da barra

    # Configurações do eixo X
    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
    ax.set_xlabel('Comprimento da Mensagem (Número de Caracteres)', fontsize=16)
    ax.set_ylabel('Mensagem', fontsize=16)
    ax.set_title('Top 30 Maiores Mensagens (Distintas) por Comprimento\n(Texto quebrado em linhas)', fontsize=18)

    # O limite do eixo X vai do zero até um pouco mais que o comprimento da mensagem mais longa
    max_length = top_longest_messages['message_length'].max()
    ax.set_xlim(0, max_length * 1.1)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    fig.tight_layout(pad=1.5)

    plt.show() # Exibe o gráfico

    # --- Salvar o Gráfico em Alta Resolução ---
    plt.savefig('top_maiores_mensagens.png', dpi=300, bbox_inches='tight')
    print("Gráfico salvo como 'top_maiores_mensagens.png' em alta resolução.")

else:
    print("Nenhuma mensagem encontrada para gerar o gráfico das maiores mensagens.")

#### 26. As 30 menores mensagens; 


In [ ]:
query = """
SELECT
    DISTINCT text_content_anonymous,
    LENGTH(text_content_anonymous) AS message_length
FROM
    mensagens
ORDER BY
    message_length ASC
LIMIT 30
"""

top_shortest_messages = conn.execute(query).fetchdf()


# --- Gerar o gráfico de barras ---
if not top_shortest_messages.empty:

    top_shortest_messages = top_shortest_messages.sort_values(by='message_length', ascending=True)

    # --- Função para quebrar linhas longas ---
    def wrap_labels(labels, width=70):
        return [textwrap.fill(label, width=width) for label in labels]

    # Aplica a quebra de linha nas mensagens que serão usadas como rótulos do eixo Y.
    wrapped_messages = wrap_labels(top_shortest_messages['text_content_anonymous'])

    # --- Ajustes para legibilidade do gráfico ---
    line_height_factor = 0.45
    line_height_padding = 0.1

    # Calcula a altura necessária para a figura
    total_text_height = sum(
        (label.count('\n') + 1) * line_height_factor + line_height_padding
        for label in wrapped_messages
    )
    fig_height = total_text_height + 4

    print(f"Altura calculada da figura (fig_height): {fig_height:.2f}")


    fig, ax = plt.subplots(figsize=(18, fig_height))


    y_pos = range(len(wrapped_messages))

    ax.barh(y_pos, top_shortest_messages['message_length'], color='skyblue', height=0.9) # Cor mudada para 'skyblue'

    ax.set_yticks(y_pos)
    ax.set_yticklabels(wrapped_messages, fontsize=11)
    ax.tick_params(axis='y', pad=10) # Padding para afastar o texto da barra

    # Configurações do eixo X
    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
    ax.set_xlabel('Comprimento da Mensagem (Número de Caracteres)', fontsize=16)
    ax.set_ylabel('Mensagem', fontsize=16)
    ax.set_title('Top 30 Menores Mensagens (Distintas) por Comprimento\n(Texto quebrado em linhas)', fontsize=18)

    max_length_in_top_30 = top_shortest_messages['message_length'].max()
    plt.xlim(0, max_length_in_top_30 * 1.1)

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    fig.tight_layout(pad=1.5)

    plt.show() # Exibe o gráfico

else:
    print("Nenhuma mensagem encontrada para gerar o gráfico das menores mensagens.")

#### 27. O dia em que foi publicado a maior quantidade de mensagens;

In [ ]:
query_mensagens_por_dia = """
SELECT
    CAST(date_message AS DATE) AS message_date,
    COUNT(id_message) AS total_messages
FROM
    mensagens
GROUP BY
    message_date
ORDER BY
    total_messages DESC
LIMIT 3
"""
mensagens_por_dia = conn.execute(query_mensagens_por_dia).fetchdf()

if not mensagens_por_dia.empty:
    mensagens_por_dia['message_date'] = pd.to_datetime(mensagens_por_dia['message_date'])
    mensagens_por_dia = mensagens_por_dia.sort_values(by='message_date')

    fig, ax = plt.subplots(figsize=(12, 6)) # Tamanho padrão para gráficos de tempo

    
    bar_width = 0.6
    ax.bar(mensagens_por_dia['message_date'], mensagens_por_dia['total_messages'],
           width=bar_width, color='teal')
    ax.set_ylabel('Número de Mensagens', fontsize=12)
    ax.set_title('Volume de Mensagens por Dia', fontsize=14)

    fig.autofmt_xdate(rotation=45) # Rotaciona os rótulos das datas
    fig_name = 'volume_mensagens_por_dia_barras.png'

    ax.set_xlabel('Data', fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

else:
    print("Não há dados de mensagens por dia para gerar o gráfico.")

#### 28. As mensagens que possuem as palavras “FACÇÃO” e “CRIMINOSA”;

In [ ]:
query_count_faccao_criminosa = """
SELECT
    COUNT(id_message) AS total_mensagens
FROM
    mensagens
WHERE
    text_content_anonymous ILIKE '%facção%' AND text_content_anonymous ILIKE '%criminosa%'
"""

count_mensagens = conn.execute(query_count_faccao_criminosa).fetchdf()
-
total_encontrado = count_mensagens.iloc[0]['total_mensagens']
print(f"Total de mensagens que contêm 'FACÇÃO' E 'CRIMINOSA': **{total_encontrado}**")

# --- Gerar o gráfico de barras ---
if total_encontrado > 0:
    fig, ax = plt.subplots(figsize=(8, 5)) # Tamanho de figura adequado para um gráfico de barra única

    # Criar uma única barra
    bar_label = 'Mensagens com "Facção" e "Criminosa"'
    ax.bar(bar_label, total_encontrado, color='darkred') # Cor representativa

    # Adicionar o valor da contagem acima da barra
    ax.text(bar_label, total_encontrado + 0.1, str(total_encontrado),
            ha='center', va='bottom', fontsize=12, weight='bold')

    ax.set_ylabel('Quantidade de Mensagens', fontsize=12)
    ax.set_title('Quantidade de Mensagens com "Facção" e "Criminosa"', fontsize=14)

    # Ajustar limite Y para garantir que o rótulo da contagem seja visível
    ax.set_ylim(0, total_encontrado * 1.2) # Adiciona 20% de margem no topo

    ax.tick_params(axis='x', labelsize=11) # Ajusta tamanho do label do eixo X
    ax.tick_params(axis='y', labelsize=11) # Ajusta tamanho do label do eixo Y
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()
else:
    print("Nenhuma mensagem encontrada para gerar o gráfico.")


#### 29. Quantidade de mensagens por dia e hora; 

In [ ]:

# --- 4. Consulta SQL para obter a quantidade de mensagens por dia e hora,
#       apenas para os 10 dias com mais mensagens ---
print("Executando consulta SQL para obter dados dos top 10 dias...")
query_daily_hourly_top_days = """
WITH Top10Days AS (
    SELECT
        CAST(date_message AS DATE) AS msg_date
    FROM
        mensagens
    GROUP BY
        msg_date
    ORDER BY
        COUNT(id_message) DESC
    LIMIT 10
)
SELECT
    CAST(m.date_message AS DATE) AS message_date,
    EXTRACT(HOUR FROM CAST(m.date_message AS TIMESTAMP)) AS message_hour,
    COUNT(m.id_message) AS total_messages
FROM
    mensagens m
WHERE
    CAST(m.date_message AS DATE) IN (SELECT msg_date FROM Top10Days)
GROUP BY
    message_date, message_hour
ORDER BY
    message_date ASC, message_hour ASC;
"""

daily_hourly_top_days_data = conn.execute(query_daily_hourly_top_days).fetchdf()
print(f"Consulta SQL concluída. Encontradas {len(daily_hourly_top_days_data)} combinações de dia/hora.")

# --- 5. Exibir a tabela detalhada (opcional, para conferência) ---
if not daily_hourly_top_days_data.empty:
    print("\n--- Quantidade de Mensagens por Hora para os Top 10 Dias Mais Ativos: ---")
    print(daily_hourly_top_days_data.to_string(index=False))
    print("-" * 50)
else:
    print("Nenhum dado encontrado para os top 10 dias mais ativos.")


# --- 6. Preparar os dados para o Heatmap ---
if not daily_hourly_top_days_data.empty:
    print("Preparando dados para o heatmap...")
    # Pivotar a tabela para o formato de heatmap: datas nas linhas, horas nas colunas
    heatmap_data = daily_hourly_top_days_data.pivot_table(
        index='message_date',
        columns='message_hour',
        values='total_messages'
    ).fillna(0) # Preenche horas sem mensagens com 0

    # Garante que todas as 24 horas (0-23) estejam nas colunas, mesmo que não haja dados
    all_hours = pd.Index(range(24), name='message_hour')
    heatmap_data = heatmap_data.reindex(columns=all_hours, fill_value=0)

    # Ordena as datas para o heatmap
    heatmap_data = heatmap_data.sort_index(ascending=True)

    print(f"Dados do heatmap preparados. Dimensões: {heatmap_data.shape}")

    # --- 7. Gerar o Heatmap ---
    print("Gerando o heatmap...")
    plt.figure(figsize=(18, max(5, len(heatmap_data) * 0.7))) # Tamanho ajustado dinamicamente

    # Criar o mapa de calor - REMOVIDO annot=True para otimização
    sns.heatmap(heatmap_data,
                cmap='viridis',
                linewidths=.5,
                cbar_kws={'label': 'Número de Mensagens'})

    plt.xlabel('Hora do Dia (0-23h)', fontsize=14)
    plt.ylabel('Data', fontsize=14)
    plt.title('Volume de Mensagens por Hora nos Top 10 Dias Mais Ativos (Sem Números)', fontsize=16)
    plt.xticks(ticks=range(24), labels=[f'{h:02d}h' for h in range(24)], rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.tight_layout() # Ajusta o layout para evitar sobreposição

    plt.show() # Exibe o gráfico (pode ser lento dependendo do ambiente)
    plt.close() # Fecha o gráfico para liberar memória
    print("Processo de geração de gráfico concluído.")
else:
    print("Não há dados suficientes para gerar o heatmap dos top 10 dias por hora.")




In [ ]:
#### PLUS Mensagens agrupadas por dia
query_top_10_days = """
SELECT
    CAST(date_message AS DATE) AS message_date,
    COUNT(id_message) AS total_messages
FROM
    mensagens
GROUP BY
    message_date
ORDER BY
    total_messages DESC
LIMIT 10
"""

top_10_days_data = conn.execute(query_top_10_days).fetchdf()

# --- 6. Gerar o gráfico de barras dos top 10 dias ---
if not top_10_days_data.empty:
    # Garantir que as datas estejam ordenadas para o gráfico, geralmente as barras
    # são plotadas na ordem em que aparecem no DataFrame.
    # Podemos reordenar por data ascendente para visualização ou manter por contagem.
    # Para "mais mensagens", manter a ordem DESC por contagem é mais direto.
    top_10_days_data = top_10_days_data.sort_values(by='total_messages', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 7)) # Tamanho ajustado para 10 barras

    # Criar o gráfico de barras horizontais
    bars = ax.barh(top_10_days_data['message_date'].astype(str),
                   top_10_days_data['total_messages'],
                   color='darkgreen', height=0.7) # Ajuste a altura se necessário

    ax.set_xlabel('Quantidade de Mensagens', fontsize=14)
    ax.set_ylabel('Data', fontsize=14)
    ax.set_title('Top 10 Dias com a Maior Quantidade de Mensagens', fontsize=16)

    # Adicionar rótulos de dados nas barras
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
                f'{int(width)}', va='center', ha='left', fontsize=10)

    # Ajustar limites do eixo X para acomodar os rótulos de dados
    max_messages = top_10_days_data['total_messages'].max()
    ax.set_xlim(0, max_messages * 1.15) # Adiciona 15% de margem

    ax.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Não há dados para gerar o gráfico dos 10 dias com mais mensagens.")

#### 30. Quantidade de mensagens por hora; 


In [ ]:
# --- Gerar o gráfico de tendência horária agregada (somando todos os dias) ---
query_hourly_trend = """
SELECT
    -- CORREÇÃO: Explicitamente converter para TIMESTAMP antes de EXTRACT
    EXTRACT(HOUR FROM CAST(date_message AS TIMESTAMP)) AS message_hour,
    COUNT(id_message) AS total_messages
FROM
    mensagens
GROUP BY
    message_hour
ORDER BY
    message_hour ASC
"""

hourly_trend_data = conn.execute(query_hourly_trend).fetchdf()

if not hourly_trend_data.empty:
    fig, ax = plt.subplots(figsize=(14, 7))

    ax.plot(hourly_trend_data['message_hour'], hourly_trend_data['total_messages'],
            marker='o', linestyle='-', color='indigo', linewidth=2, markersize=8)

    ax.set_xlabel('Hora do Dia (0-23h)', fontsize=14)
    ax.set_ylabel('Número Total de Mensagens', fontsize=14)
    ax.set_title('Volume Total de Mensagens por Hora do Dia', fontsize=16)

    # Definir os ticks do eixo X para cada hora completa
    ax.set_xticks(range(24))
    ax.set_xticklabels([f'{h:02d}h' for h in range(24)], rotation=45, ha='right', fontsize=11)

    ax.grid(True, linestyle='--', alpha=0.7)

    # Adicionar rótulos de dados em cada ponto
    for x, y in zip(hourly_trend_data['message_hour'], hourly_trend_data['total_messages']):
        ax.text(x, y + (max(hourly_trend_data['total_messages']) * 0.03), str(y),
                ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()
else:
    print("Não há dados para gerar o gráfico de volume de mensagens por hora do dia.")


#### 31. A nuvem de palavras referente às mensagens de texto (após a remoção de stop words);

In [ ]:
from wordcloud import WordCloud, STOPWORDS as EN_STOPWORDS
import nltk
from nltk.corpus import stopwords
import re

# --- Download de recursos do NLTK (execute uma vez se necessário) ---
try:
    stopwords.words('portuguese')
except LookupError:
    print("Baixando 'stopwords' do NLTK...")
    nltk.download('stopwords')
# --- Fim do download ---


# 1. Concatenar todos os textos e limpeza básica
if 'text_content_anonymous' not in df.columns or df['text_content_anonymous'].dropna().empty:
    print("Coluna 'text_content_anonymous' não encontrada ou vazia após remover NaNs. Nuvem de palavras não gerada.")
else:
    # Concatenar todos os textos, convertendo para string e tratando NaNs
    all_text = " ".join(df['text_content_anonymous'].astype(str).dropna())

    if not all_text.strip():
        print("Nenhum texto encontrado para gerar a nuvem de palavras.")
    else:
        all_text_lower = all_text.lower()

        # 3. Remover pontuações básicas (mantendo letras, números e espaços)
        all_text_no_punct = re.sub(r'[^\w\s]', '', all_text_lower)


        # 4. Remover Stop Words em Português
        stop_words_pt = set(stopwords.words('portuguese'))
        
        # Adicionar stop words personalizadas
        custom_stopwords = {'pra', 'vc', 'tbm', 'q', 'né', 'aí', 'tá', 'https', 'http', 'www', 
                            'com', 'br', 'youtube', 'instagram', 'facebook', 'twitter', 'tiktok',
                            'video', 'imagem', 'foto', 'link', 'post'} # Adicione mais conforme necessário
        
        all_stopwords = stop_words_pt.union(custom_stopwords)
        
        # 5. Gerar a Nuvem de Palavras
        print("Gerando a nuvem de palavras...")
        try:
            wordcloud = WordCloud(
                width=1200, 
                height=600, 
                background_color='white', 
                stopwords=all_stopwords, # Passa o conjunto de stopwords
                min_font_size=10,
                max_words=200,          # Número máximo de palavras na nuvem
                colormap='viridis',     # Esquema de cores
                collocations=True       # Tenta incluir bigramas (pares de palavras) comuns
            ).generate(all_text_no_punct) # Gera a partir do texto pré-processado

            # Exibir a nuvem de palavras
            plt.figure(figsize=(15, 7))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis("off") # Remover eixos
            plt.title("Nuvem de Palavras das Mensagens de Texto (após remoção de stop words)", fontsize=16, pad=10)
            plt.tight_layout(pad=0)
            plt.show()
            
        except ValueError as e:
            if "empty" in str(e).lower():
                 print(f"Erro ao gerar a nuvem de palavras: Não há palavras suficientes após o pré-processamento e remoção de stopwords. Detalhe: {e}")
            else:
                print(f"Erro ao gerar a nuvem de palavras: {e}")
        except Exception as e:
            print(f"Um erro inesperado ocorreu ao gerar a nuvem de palavras: {e}")


#### 32. A rede interativa das palavras referente às mensagens de texto (após a remoção de stop words); 

In [ ]:
import io
import nltk
from nltk.corpus import stopwords
from collections import defaultdict
import itertools
import networkx as nx
from pyvis.network import Network

messages = df['text_content_anonymous'].dropna().tolist()
stopwords_pt = set(stopwords.words('portuguese'))

def preprocess_text(text):
    """Function to clean and tokenize text for network analysis."""
    text = text.lower()  # Convert to lowercase
    # Remove punctuation, numbers, and special characters, keeping accented characters
    text = re.sub(r'[^a-záéíóúãõâêôàçü\s]', '', text)
    words = text.split()  # Tokenize into words
    # Remove stop words and very short words (2 or fewer characters)
    words = [word for word in words if word not in stopwords_pt and len(word) > 2]
    return words

processed_messages = [preprocess_text(msg) for msg in messages]


co_occurrence = defaultdict(int)  
word_counts = defaultdict(int)   

for msg_words in processed_messages:
    for word in msg_words:
        word_counts[word] += 1
    
    for word1, word2 in itertools.combinations(sorted(msg_words), 2):
        co_occurrence[(word1, word2)] += 1

G = nx.Graph()

min_word_frequency = 2 
min_co_occurrence = 2  

for word, count in word_counts.items():
    if count >= min_word_frequency:
        G.add_node(word, size=count * 3, title=f"Ocorrências: {count}") 

for (word1, word2), count in co_occurrence.items():
    if count >= min_co_occurrence and G.has_node(word1) and G.has_node(word2):
        G.add_edge(word1, word2, weight=count, title=f"Co-ocorrências: {count}")

print("Generating interactive word network graph. This might take a few moments...")

net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white", notebook=True, cdn_resources='remote')

net.show_buttons(filter_=['physics']) 

net.from_nx(G)

output_filename = "rede_interativa_palavras.html"
net.show(output_filename)

print(f"\n--- Interactive word network generated successfully! ---")
print(f"Open the file '{output_filename}' in your web browser to explore the network.")
print("You can drag nodes, zoom in/out, and interact with the graph to discover word relationships.")

#### 33. Proporção de mensagens com e sem URL;

In [ ]:
# 1. Contar as ocorrências de True e False na coluna 'has_media_url'
url_presence_counts = df['has_media_url'].value_counts()

count_com_url = url_presence_counts.get(True, 0)  # has_media_url == True
count_sem_url = url_presence_counts.get(False, 0) # has_media_url == False

if count_com_url + count_sem_url == 0:
    print("Nenhuma mensagem encontrada para analisar a presença de URLs.")
else:
    labels = ['Com URL\n(has_media_url=True)', 'Sem URL\n(has_media_url=False)']
    sizes = [count_com_url, count_sem_url]
    
    # Cores para o gráfico
    colors = ['#66b3ff', '#ff9999'] # Azul claro e Rosa claro

    # Explodir uma fatia (opcional, se quiser destacar uma)
    explode = (0.05, 0) if count_com_url > count_sem_url else (0, 0.05) # Explode a maior fatia

    # 2. Criar o gráfico de pizza
    fig, ax = plt.subplots(figsize=(8, 8))
    
    wedges, texts, autotexts = ax.pie(
        sizes, 
        explode=explode, 
        labels=None, # Rótulos serão adicionados na legenda para não poluir o donut
        colors=colors, 
        autopct='%1.1f%%', # Formato da porcentagem
        startangle=90,
        wedgeprops=dict(width=0.4, edgecolor='w') 
    )


    # Melhorar a aparência dos textos de porcentagem
    for autotext in autotexts:
        autotext.set_color('black')
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')

    ax.axis('equal')  # Assegura que a pizza seja desenhada como um círculo.
    
    plt.title('Proporção de Mensagens Com vs. Sem URL', fontsize=15, pad=20)
    
    # Adicionar legenda com os rótulos corretos
    # Usar os tamanhos para mostrar as contagens absolutas na legenda
    legend_labels = [f'{l} ({s})' for l, s in zip(labels, sizes)]
    ax.legend(wedges, legend_labels,
              title="Categoria",
              loc="center left",
              bbox_to_anchor=(1, 0, 0.5, 1), # Posicionar legenda à direita
              fontsize=10)

    plt.tight_layout(rect=[0, 0, 0.8, 1]) # Ajustar layout para a legenda não ser cortada
    plt.show()

#### 34. Proporção de desinformação;

In [ ]:
# --- Classificar mensagens com base na regra do score_misinformation ---
def classify_misinformation(score):
    if score > 0.5:
        return 'Desinformação'
    elif score == 0.5:
        return 'Neutro'
    else: # score < 0.5
        return 'Normal'

df['misinformation_category'] = df['score_misinformation'].apply(classify_misinformation)

# Contar o número de mensagens por categoria
misinformation_counts = df['misinformation_category'].value_counts()

# Reordenar as categorias para o gráfico, se desejar (opcional, mas bom para consistência)
order = ['Normal', 'Neutro', 'Desinformação']
misinformation_counts = misinformation_counts.reindex(order, fill_value=0) # fill_value=0 para garantir que todas as categorias apareçam

# Calcular as proporções
total_messages = misinformation_counts.sum()
misinformation_proportions = misinformation_counts / total_messages

print("--- Proporção de Mensagens por Categoria de Desinformação ---")
print(f"Total de mensagens: {total_messages}")
print("\nContagem:")
print(misinformation_counts)
print("\nProporção:")
print(misinformation_proportions.apply(lambda x: f"{x:.2%}"))

# --- Gerar o Gráfico de Pizza ---
plt.figure(figsize=(9, 7)) # Aumenta um pouco o tamanho da figura

# Define cores para cada categoria
colors = {
    'Normal': '#66b3ff',      # Azul claro
    'Neutro': '#ffcc99',      # Laranja claro
    'Desinformação': '#ff6666' # Vermelho
}
plot_colors = [colors[cat] for cat in misinformation_proportions.index]

wedge_props = {'linewidth': 1, 'edgecolor': 'white'} # Borda das fatias

wedges, texts, autotexts = plt.pie(
    misinformation_proportions,
    labels=misinformation_proportions.index,
    autopct='%1.1f%%',
    
    startangle=90,
    colors=plot_colors,
    wedgeprops=wedge_props,
    textprops={'color': 'white'} # Cor do label
)

for autotext in autotexts:
    autotext.set_color('black') # Cor preta para legibilidade
    autotext.set_fontsize(10)
    autotext.set_weight('bold')

plt.title('Proporção de Mensagens por Nível de Desinformação', fontsize=16, pad=20)
plt.axis('equal') # Garante que o gráfico de pizza seja um círculo
plt.tight_layout() # Ajusta o layout para evitar sobreposição

plt.show()


#### 35. Proporção de mensagens contendo mídia e desinformação;

In [ ]:
df['score_misinformation'] = df['score_misinformation'].fillna(0.0)

# 1. Filtrar as mensagens que têm desinformação 
df_misinformation = df[df['score_misinformation'] > 0.5].copy()

# Se não houver mensagens de desinformação, avisa e para.
if df_misinformation.empty:
    print("Não há mensagens classificadas como 'Desinformação' (score > 0.5) neste dataset para analisar.")
else:
    # --- 2. Dentro desse subconjunto, contar as que têm mídia e as que não têm ---
    media_in_misinformation_counts = df_misinformation['has_media'].value_counts()

    # Renomear os índices para algo mais legível no gráfico
    media_in_misinformation_counts.index = media_in_misinformation_counts.index.map(
        {True: 'Com Mídia', False: 'Sem Mídia'}
    )

    # Reordenar para o gráfico se desejar (opcional)
    order = ['Com Mídia', 'Sem Mídia']
    media_in_misinformation_counts = media_in_misinformation_counts.reindex(order, fill_value=0)

    # Calcular as proporções dentro do universo da desinformação
    total_misinformation_messages = media_in_misinformation_counts.sum()
    media_in_misinformation_proportions = media_in_misinformation_counts / total_misinformation_messages

    print(f"--- Análise de Mídia em Mensagens de Desinformação ---")
    print(f"Total de mensagens de Desinformação (score > 0.5): {total_misinformation_messages}")
    print("\nContagem de Mídia nestas mensagens:")
    print(media_in_misinformation_counts)
    print("\nProporção de Mídia nestas mensagens:")
    print(media_in_misinformation_proportions.apply(lambda x: f"{x:.2%}"))

    # --- Gerar o Gráfico de Pizza ---
    plt.figure(figsize=(8, 6))

    # Definir cores para as fatias
    colors = {
        'Com Mídia': '#ff6666',  # Vermelho para "Com Mídia" na desinformação
        'Sem Mídia': '#66b3ff'   # Azul para "Sem Mídia" na desinformação
    }
    plot_colors = [colors[cat] for cat in media_in_misinformation_proportions.index]


    wedge_props = {'linewidth': 1, 'edgecolor': 'white'} # Borda das fatias

    wedges, texts, autotexts = plt.pie(
        media_in_misinformation_proportions,
        labels=media_in_misinformation_proportions.index,
        autopct='%1.1f%%',
        startangle=90,
        colors=plot_colors,
        wedgeprops=wedge_props,
        textprops={'color': 'white'} # Cor do label
    )

    # Ajusta a cor do texto dos percentuais para ser visível no fundo da fatia
    for autotext in autotexts:
        autotext.set_color('black')
        autotext.set_fontsize(10)
        autotext.set_weight('bold')

    plt.title('Proporção de Mídia em Mensagens de Desinformação', fontsize=16, pad=20)
    plt.axis('equal') # Garante que o gráfico de pizza seja um círculo
    plt.tight_layout()
    plt.show()

    print(f"\n--- Gráfico gerado com sucesso! ---")
    print(f"O gráfico foi salvo como '{plot_filename}'.")

#### 36. Distribuição de mensagens por score de desinformação;

In [ ]:

# Executar a consulta SQL para agrupar por score_misinformation e contar, usando a tabela 'mensagens'
query = """
SELECT
    score_misinformation,
    COUNT(*) AS count_messages
FROM
    mensagens
GROUP BY
    score_misinformation
ORDER BY
    score_misinformation ASC;
"""

result_df = conn.execute(query).fetchdf()

## Gráfico de Barras Simplificado (Monocromático)

plt.figure(figsize=(12, 7))

# Plotar o gráfico de barras 
sns.barplot(x='score_misinformation', y='count_messages', data=result_df, color='steelblue')

plt.title('Distribuição de Mensagens por Score de Desinformação', fontsize=16, pad=20)
plt.xlabel('Score de Desinformação', fontsize=12)
plt.ylabel('Número de Mensagens', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


#### 37. Proporção de sentimentos;